<a href="https://colab.research.google.com/github/lamonega/colab-llm/blob/main/colab_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import subprocess
import threading
import time

import requests

# ── Configuración ──────────────────────────────────────────────────────────────

MODEL_NAME = "qwen3:14b"  # Cambiar si se desea otro modelo

os.environ["OLLAMA_HOST"]           = "0.0.0.0"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"
os.environ["OLLAMA_KEEP_ALIVE"]     = "-1"
os.environ["OLLAMA_ORIGINS"]        = "*"  # Evita problemas de CORS en entornos externos
os.environ["MODEL_NAME"]            = MODEL_NAME

# ── Funciones auxiliares ───────────────────────────────────────────────────────

def command_exists(cmd):
    return shutil.which(cmd) is not None

def ollama_responds():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

# ── 1. Dependencias del sistema ────────────────────────────────────────────────

# curl
if command_exists("curl"):
    print("[OK] curl ya instalado.")
else:
    print("Instalando curl...")
    subprocess.run(["apt-get", "update"], capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "curl"], capture_output=True, check=True)
    print("[OK] curl instalado.")

# zstd
if command_exists("zstd"):
    print("[OK] zstd ya instalado.")
else:
    print("Instalando zstd...")
    subprocess.run(["apt-get", "update"], capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], capture_output=True, check=True)
    print("[OK] zstd instalado.")

# lshw
if command_exists("lshw"):
    print("[OK] lshw ya instalado.")
else:
    print("Instalando lshw...")
    subprocess.run(["apt-get", "update"], capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "lshw"], capture_output=True, check=True)
    print("[OK] lshw instalado.")

# pciutils
if command_exists("pciutils"):
    print("[OK] pciutils ya instalado.")
else:
    print("Instalando pciutils...")
    subprocess.run(["apt-get", "update"], capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "pciutils"], capture_output=True, check=True)
    print("[OK] pciutils instalado.")

# Ollama
if command_exists("ollama"):
    print("[OK] Ollama ya instalado.")
else:
    print("Instalando Ollama...")

    # Paso 1: descargar el script
    curl = subprocess.run(
        ["curl", "-fsSL", "https://ollama.com/install.sh"],
        capture_output=True, text=True
    )
    if curl.returncode != 0:
        raise RuntimeError(f"[ERROR] No se pudo descargar el instalador:\n{curl.stderr}")

    # Paso 2: ejecutar el script descargado
    install = subprocess.run(
        ["sh"], input=curl.stdout, capture_output=True, text=True
    )
    if install.returncode != 0:
        raise RuntimeError(f"[ERROR] Falló la instalación de Ollama:\n{install.stderr}")

    print("[OK] Ollama instalado.")

# ── 2. Servicio Ollama ─────────────────────────────────────────────────────────

# Si ya responde, no hace falta reiniciar
if ollama_responds():
    print("[OK] Servicio Ollama ya activo.")
else:
    print("Iniciando servicio Ollama...")
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(2)

    def _serve():
        subprocess.call(["ollama", "serve"])

    threading.Thread(target=_serve, daemon=True).start()

    for i in range(30):
        if ollama_responds():
            print(f"[OK] Servicio listo en {i + 1}s.")
            break
        time.sleep(1)
    else:
        raise RuntimeError("[ERROR] Ollama no arrancó en 30s.")

# ── 3. Modelo ──────────────────────────────────────────────────────────────────

# Descargar solo si no está disponible localmente
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME in result.stdout:
    print(f"[OK] Modelo '{MODEL_NAME}' ya disponible.")
else:
    print(f"Descargando {MODEL_NAME}...")
    pull = subprocess.run(["ollama", "pull", MODEL_NAME], capture_output=True, text=True)
    if pull.returncode != 0:
        raise RuntimeError(f"[ERROR] Falló la descarga: {pull.stderr}")
    print(f"[OK] Modelo descargado: {MODEL_NAME}")

# Cargar en VRAM con una inferencia inicial
print("Cargando modelo en VRAM...")
try:
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": MODEL_NAME, "prompt": "Hola", "stream": False},
        timeout=300,
    )
    if r.status_code == 200:
        print("[OK] Modelo listo en VRAM.")
    else:
        print(f"[ADVERTENCIA] Carga inicial respondió con código {r.status_code}.")
except Exception as e:
    print(f"[ADVERTENCIA] No se pudo cargar el modelo: {e}")

# ──────────────────────────────────────────────────────────────────────────────
print("[OK] Celda 1 completada.")

In [ ]:
import os
import re
import shutil
import subprocess
import time

import requests

# ── Validación previa ──────────────────────────────────────────────────────────

model = os.environ.get("MODEL_NAME")
if not model:
    raise RuntimeError("MODEL_NAME no definido. Ejecutá la celda 1 primero.")

print(f"Usando modelo: {model}")

# ── Funciones auxiliares ───────────────────────────────────────────────────────

def command_exists(cmd):
    return shutil.which(cmd) is not None

def ollama_responds():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

def tunnel_active():
    try:
        result = subprocess.run(["pgrep", "-f", "cloudflared tunnel"], capture_output=True, text=True)
        return result.returncode == 0
    except Exception:
        return False

# ── 1. Dependencias del sistema ────────────────────────────────────────────────

# wget
if command_exists("wget"):
    print("[OK] wget ya instalado.")
else:
    print("Instalando wget...")
    subprocess.run(["apt-get", "update"], capture_output=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "wget"], capture_output=True, check=True)
    print("[OK] wget instalado.")

# cloudflared
if os.path.exists("./cloudflared"):
    print("[OK] cloudflared ya disponible.")
else:
    print("Descargando cloudflared...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared",
    ], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
    print("[OK] cloudflared listo.")

# ── 2. Servicio Ollama ─────────────────────────────────────────────────────────

if not ollama_responds():
    raise RuntimeError("Ollama local no responde. Revisá la celda 1.")

print("[OK] Servicio Ollama local activo.")

# ── 3. Túnel Cloudflare ────────────────────────────────────────────────────────

# Detener túnel previo si existe
if tunnel_active():
    print("Deteniendo túnel anterior...")
    subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
    time.sleep(2)

# Iniciar túnel
print("Iniciando túnel Cloudflare...")
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Extraer URL pública del output del proceso
public_url = None
start_time = time.time()

for line in proc.stdout:
    if "https://" in line and "trycloudflare.com" in line:
        match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
        if match:
            public_url = match.group(1)
            break
    if time.time() - start_time > 60:
        proc.terminate()
        raise RuntimeError("No se obtuvo URL pública en 60s.")

if not public_url:
    proc.terminate()
    raise RuntimeError("No se pudo extraer la URL pública.")

print(f"[OK] URL pública: {public_url}")

# ── 4. Verificación ────────────────────────────────────────────────────────────

# Verificar acceso externo
print("Verificando acceso público...")
for intento in range(1, 13):
    try:
        resp = requests.get(f"{public_url}/api/tags", timeout=5)
        if resp.status_code == 200:
            print(f"[OK] URL accesible (intento {intento}).")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    proc.terminate()
    raise RuntimeError("La URL pública no responde tras 60s.")

# Probar modelo
print(f"Probando modelo '{model}'...")
try:
    r = requests.post(
        f"{public_url}/api/generate",
        json={"model": model, "prompt": "Responde solo con el número: 2+2", "stream": False},
        timeout=120,
    )
    r.raise_for_status()
    respuesta = r.json().get("response", "").strip()
    print(f"[OK] Modelo responde: {respuesta}")
except Exception as e:
    proc.terminate()
    raise RuntimeError(f"Error al probar el modelo: {e}")

# ── Resumen ────────────────────────────────────────────────────────────────────

print("\n" + "=" * 50)
print(f"URL pública: {public_url}")
print(f"Modelo:      {model}")
print("=" * 50)

proc.wait()